# DFT practice — Lessons 3, 4, 5

| Lesson | Question | Molecule(s) |
|---|---|---|
| **3. Orbitals** | What do HOMO/LUMO energies mean, and what *don't* they mean? | benzene |
| **4. From SMILES** | How does a string of characters become a molecule PySCF can compute? | benzene, anthraquinone |
| **5. Charge & solvent** | Why must eosin Y be computed as −2, and why in water? | phenol, phenolate |

**How to use this notebook**
1. Run the setup cell first.
2. In each exercise cell, replace the lines marked `# TODO` with your own code, then run it.
3. Run the **Check** cell under it. It tells you whether your numbers are in the right range.
4. Answer the **Think** questions in your own words (double-click a markdown cell to type).
5. Stuck for more than 10 minutes? The solutions are at the very bottom (collapsed).

Everything uses **B3LYP / def2-SVP**, the same level as the photocatalyst project.
Each calculation takes seconds to a couple of minutes on a laptop.

In [ ]:
# 0. Setup — run once. (Restart the kernel after a first-time install.)
# %pip install -q pyscf rdkit py3Dmol

import importlib, sys, types
def _import_pyscf():
    """Same workaround as utils/common.py: on some Macs pyscf-dispersion is broken and makes
    `import pyscf` crash. If so, replace it with an empty module (we never use D3 here)."""
    try:
        mod = importlib.import_module("pyscf")
        from pyscf.dispersion import dftd3  # noqa: F401
        return mod
    except ImportError:
        return importlib.import_module("pyscf")
    except TypeError:
        for k in [k for k in sys.modules if k == "pyscf" or k.startswith("pyscf.")]:
            del sys.modules[k]
        sys.modules["pyscf.dispersion"] = types.ModuleType("pyscf.dispersion")
        return importlib.import_module("pyscf")
_import_pyscf()

from pyscf import gto, scf, dft, lib
import numpy as np
lib.num_threads(8)          # use more if your machine has them
H2EV = 27.211386            # 1 Hartree in eV
print("ready")

---
# Lesson 3 — Orbitals: what they mean, and what they don't

**Background.** A Kohn–Sham DFT calculation gives you a list of orbital energies `mf.mo_energy`
(in Hartree, lowest first) and their occupations `mf.mo_occ` (2 = doubly occupied, 0 = empty).
For a closed-shell molecule with *N* electrons there are *N*/2 occupied orbitals, so
(counting from 0, as Python does):

- HOMO = orbital number `N//2 - 1`
- LUMO = orbital number `N//2`

It is tempting to read −HOMO as the ionisation energy (IE). That is **Koopmans' theorem**. It works
roughly for Hartree–Fock but not for approximate DFT functionals. In this lesson you will measure
how big the error is, and compute the IE properly instead.

Benzene geometry below: B3LYP/def2-SVP optimised, from `results/opt/test_benzene/`.

In [ ]:
BENZENE_XYZ = """
12
test_benzene b3lyp/def2-svp disp=d3bj solvent=None charge=0 spin=0 E=-232.10359482 Eh
C     -1.25095048     0.62435612    -0.02844074
C     -1.16585107    -0.77068698     0.01156658
C      0.08470476    -1.39543975     0.04002334
C      1.25094625    -0.62435479     0.02844069
C      1.16585548     0.77068820    -0.01156654
C     -0.08470372     1.39543590    -0.04002319
H     -2.22876326     1.11193177    -0.05059728
H     -2.07752572    -1.37324716     0.02065739
H      0.15053283    -2.48586679     0.07128161
H      2.22875795    -1.11192925     0.05059714
H      2.07752985     1.37324926    -0.02065742
H     -0.15053240     2.48586411    -0.07128153
"""
benzene_atoms = "\n".join(BENZENE_XYZ.strip().splitlines()[2:])   # drop the 2 header lines
print(benzene_atoms.splitlines()[0], "...", len(benzene_atoms.splitlines()), "atoms")

### Exercise 3.1 — orbital energies of benzene

In [ ]:
# Build the molecule and run B3LYP/def2-SVP
mol = gto.M(atom=benzene_atoms, basis="def2-svp", verbose=0)
mf = None          # TODO: create a dft.RKS object with xc="b3lyp"
E_neutral = None   # TODO: run the SCF (mf.kernel()) and store the total energy

n_occ = None       # TODO: number of doubly occupied orbitals (use mol.nelectron)

# print HOMO-2 ... LUMO+1 in eV with a label
for k in range(n_occ - 3, n_occ + 2):
    label = "occ " if mf.mo_occ[k] > 0 else "virt"
    print(k, label, round(mf.mo_energy[k] * H2EV, 3), "eV")
HOMO = mf.mo_energy[n_occ - 1] * H2EV
LUMO = mf.mo_energy[n_occ] * H2EV

In [ ]:
# Check 3.1
assert mol.nelectron == 42 and n_occ == 21, "benzene C6H6 has 6*6 + 6*1 = 42 electrons"
assert abs(HOMO - (-6.97)) < 0.05, f"HOMO should be about -6.97 eV, got {HOMO:.2f}"
assert abs(LUMO - (-0.25)) < 0.05, f"LUMO should be about -0.25 eV, got {LUMO:.2f}"
print(f"OK  HOMO {HOMO:.3f} eV, LUMO {LUMO:.3f} eV, gap {LUMO-HOMO:.3f} eV")

**Think 3.1**
- Two pairs of orbitals have (almost) the same energy. Which ones? What does that mean physically
  for a molecule with six-fold symmetry?
- *Your answer:* …

### Exercise 3.2 — ionisation energy: Koopmans vs ΔSCF

In [ ]:
# ΔSCF: IE = E(cation) - E(neutral), each from its own SCF.
# The cation C6H6+ has 41 electrons: one unpaired electron -> spin = 1 (PySCF's spin = N_alpha - N_beta)
# and it must be an *unrestricted* calculation (dft.UKS), because alpha and beta electrons now differ.
cation = None      # TODO: gto.M(... same atoms and basis ..., charge=?, spin=?, verbose=0)
mf_cat = None      # TODO: dft.UKS(...) with xc="b3lyp"
E_cation = None    # TODO: run it

IE_dSCF = None     # TODO: in eV
IE_koopmans_B3LYP = -HOMO

# For comparison: Hartree-Fock's -HOMO (Koopmans' theorem is "meant" for HF)
hf = scf.RHF(mol); hf.kernel()
IE_koopmans_HF = -hf.mo_energy[n_occ - 1] * H2EV

print(f"experiment            9.24 eV")
print(f"-HOMO (B3LYP)       {IE_koopmans_B3LYP:6.2f} eV")
print(f"-HOMO (HF)          {IE_koopmans_HF:6.2f} eV")
print(f"dSCF  (B3LYP)       {IE_dSCF:6.2f} eV")

In [ ]:
# Check 3.2
assert abs(IE_dSCF - 9.24) < 0.3, f"dSCF IE should be close to 9.2 eV, got {IE_dSCF:.2f}"
print("OK  B3LYP -HOMO misses the IE by", round(9.24 - IE_koopmans_B3LYP, 2), "eV; dSCF misses it by",
      round(abs(9.24 - IE_dSCF), 2), "eV")

**Think 3.2**
- Why is the B3LYP HOMO so far off, while the ΔSCF number is so good? (Hint: in ΔSCF you compare
  two *total* energies, so the error of approximate exchange largely cancels. The orbital energy
  inherits that error, known as self-interaction error, directly.)
- HF's −HOMO is much better than B3LYP's. Why might that be?
- So when the project's diagram plots B3LYP HOMO energies, what can and can't we read from it?
- *Your answer:* …

### Exercise 3.3 — look at the orbitals (optional, needs py3Dmol)

In [ ]:
from pyscf.tools import cubegen
import py3Dmol

def show_orbital(k, iso=0.03):
    cubegen.orbital(mol, "tmp.cube", mf.mo_coeff[:, k], nx=50, ny=50, nz=50)
    cube = open("tmp.cube").read()
    v = py3Dmol.view(width=350, height=300)
    v.addModel(cube, "cube"); v.setStyle({"stick": {"radius": 0.1}})
    v.addVolumetricData(cube, "cube", {"isoval": iso, "color": "blue", "opacity": 0.8})
    v.addVolumetricData(cube, "cube", {"isoval": -iso, "color": "red", "opacity": 0.8})
    v.zoomTo(); return v

# TODO: show HOMO, HOMO-1 and LUMO (use n_occ to get their indices) and count the nodal planes
show_orbital(None).show()

**Think 3.3** How many nodal planes cut through the ring in the HOMO? In the LUMO? How does that
fit the particle-on-a-ring picture? *Your answer:* …

### Exercise 3.4 (bonus) — the optical gap needs TDDFT

In [ ]:
from pyscf import tddft
td = None              # TODO: tddft.TDDFT(mf), ask for 3 states (td.nstates = 3), run td.kernel()

for i, (e, f) in enumerate(zip(td.e * H2EV, td.oscillator_strength())):
    print(f"S{i+1}: {e:.2f} eV  ({1239.84/e:.0f} nm)   oscillator strength f = {f:.3f}")
print(f"HOMO-LUMO gap: {LUMO-HOMO:.2f} eV ;  experiment: weak band 4.9 eV (254 nm), strong band ~6.9 eV")

**Think 3.4**
- S1 has oscillator strength f ≈ 0: it is symmetry-forbidden, which is why benzene's 254 nm band
  is so weak.
- Which number is closer to the experimental absorption, the HOMO–LUMO gap or S1?
- Write down the three different "gaps" you have now met: KS gap, fundamental gap (IE − EA),
  optical gap. *Your answer:* …

---
# Lesson 4 — From SMILES to a molecule PySCF can compute

**Background.** PySCF needs **atoms + 3D coordinates + charge**. A SMILES string only gives atoms
and bonds, and the hydrogens are implicit. So there are four steps in between:

```
SMILES  --MolFromSmiles-->  graph  --AddHs-->  graph with H  --EmbedMolecule (ETKDG)-->  rough 3D
        --MMFFOptimizeMolecule-->  sensible 3D  --> list of (symbol, (x, y, z))  --> gto.M(...)
```

The project's `01_represent.py` does exactly this, plus checks and conformer search. Here you do
the bare version yourself.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
RDLogger.DisableLog("rdApp.*")

### Exercise 4.1 — read a SMILES and check it

In [ ]:
smi = "c1ccccc1"            # benzene
m = None                    # TODO: Chem.MolFromSmiles
print("formula:", None)     # TODO: CalcMolFormula(m)
print("charge :", None)     # TODO: Chem.GetFormalCharge(m)
print("heavy atoms:", m.GetNumAtoms())
Draw.MolToImage(m, size=(250, 200))

**Think 4.1** `m.GetNumAtoms()` gives 6, not 12. Why? What does `Chem.MolFromSmiles("c1ccccc")`
return (try it)? Why is that a useful thing to check before going further? *Your answer:* …

### Exercise 4.2 — write the function smiles_to_atoms

In [ ]:
def smiles_to_atoms(smiles, seed=42):
    """Return (atoms, charge) where atoms = [(symbol, (x, y, z)), ...] in Angstrom."""
    m = Chem.MolFromSmiles(smiles)
    q = Chem.GetFormalCharge(m)
    mh = None                       # TODO: add explicit hydrogens
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    # TODO: embed mh in 3D with these params (AllChem.EmbedMolecule)
    # TODO: clean it up with the MMFF force field (AllChem.MMFFOptimizeMolecule)
    pos = mh.GetConformer().GetPositions()          # N x 3 array
    atoms = None                    # TODO: build the list [(symbol, (x, y, z)), ...]
    return atoms, q

atoms, q = smiles_to_atoms("c1ccccc1")
print(len(atoms), "atoms, charge", q); print(atoms[0])

In [ ]:
# Check 4.2
assert len(atoms) == 12 and q == 0, "benzene should give 12 atoms (6 C + 6 H) and charge 0"
cc = [np.linalg.norm(np.subtract(atoms[i][1], atoms[j][1])) for i in range(12) for j in range(i+1, 12)
      if atoms[i][0] == atoms[j][0] == "C"]
assert abs(min(cc) - 1.39) < 0.03, f"shortest C-C should be ~1.39 A, got {min(cc):.3f}"
print(f"OK  MMFF C-C = {min(cc):.3f} A (DFT gave 1.398, experiment 1.397)")

### Exercise 4.3 — hand it to PySCF

In [ ]:
def single_point(atoms, charge=0, water=False):
    """B3LYP/def2-SVP single point. Returns (mol, mf, HOMO_eV, LUMO_eV)."""
    mol = None          # TODO: gto.M(atom=atoms, basis=..., charge=charge, spin=0, verbose=0)
    mf = dft.RKS(mol, xc="b3lyp").density_fit()      # density fitting = faster, tiny error
    if water:
        mf = mf.ddCOSMO(); mf.with_solvent.eps = 78.3553   # implicit water (Lesson 5)
    mf.kernel()
    n_occ = None        # TODO
    return mol, mf, mf.mo_energy[n_occ - 1] * H2EV, mf.mo_energy[n_occ] * H2EV

mol_b, mf_b, h, l = single_point(atoms)
print(f"benzene (MMFF geometry): {mol_b.nelectron} electrons, {mol_b.nao} basis functions, "
      f"HOMO {h:.3f}  LUMO {l:.3f} eV")

In [ ]:
# Check 4.3
assert mol_b.nao == 114, "def2-SVP benzene: 6 C x 14 + 6 H x 5 = 114 basis functions"
assert abs(h - HOMO) < 0.1, f"HOMO on the MMFF geometry should be within 0.1 eV of Lesson 3 ({HOMO:.2f})"
print(f"OK  MMFF vs DFT geometry changes the HOMO by {h-HOMO:+.3f} eV")

### Exercise 4.4 — anthraquinone, and compare with the project

In [ ]:
aq_smiles = None                 # TODO: anthraquinone SMILES (look it up, or use the project's csv)
aq_atoms, aq_q = smiles_to_atoms(aq_smiles)

# TODO: run single_point twice, in vacuum and in water, and print HOMO, LUMO and gap for each
# (each run takes ~1-3 min on a laptop)

print("project (DFT-optimised geometry, water): HOMO -7.314  LUMO -3.041  gap 4.273 eV")

**Think 4.4** Your water result should be within about 0.03 eV of the project's. Where does the
small remaining difference come from? (Two things differ: the geometry, MMFF vs DFT, and the
conformer search.) Is the solvent effect on this *neutral* molecule large or small? Remember the
answer for Lesson 5. *Your answer:* …

---
# Lesson 5 — Charge and solvent

**Background.** Eosin Y works in water as a **dianion**, with two negative charges. Two things can go
wrong when computing an ion:

1. **Forgetting the charge.** The code then computes a different molecule with a different number
   of electrons.
2. **Leaving out the solvent.** In vacuum, the extra electrons of an anion are barely held, or not
   held at all. In water, the surrounding polar solvent stabilises them.

Eosin Y is big (35 atoms, ~500 basis functions), so we practise on the smallest relevant
example: **phenol** (C₆H₅OH) and **phenolate** (C₆H₅O⁻), its deprotonated anion. The same chemistry
happens at eosin Y's phenol oxygen.

`ddCOSMO` places the molecule in a cavity surrounded by a dielectric continuum (ε = 78.36 for
water). It is not explicit water molecules, just their average electrostatic effect.

### Exercise 5.1 — build both and read the charge from the SMILES

In [ ]:
phenol_atoms, phenol_q = None, None          # TODO: smiles_to_atoms("Oc1ccccc1")
phenolate_atoms, phenolate_q = None, None    # TODO: the SMILES with [O-] instead of O
print("phenol   :", len(phenol_atoms), "atoms, charge", phenol_q)
print("phenolate:", len(phenolate_atoms), "atoms, charge", phenolate_q)

In [ ]:
# Check 5.1
assert (len(phenol_atoms), phenol_q) == (13, 0) and (len(phenolate_atoms), phenolate_q) == (12, -1)
print("OK  the [O-] in the SMILES removed one H and set the charge to -1")

### Exercise 5.2 — what happens if you forget the charge?

In [ ]:
try:
    # TODO: build phenolate with gto.M but WITHOUT charge=-1 (leave the default 0), spin=0
    pass
except Exception as err:
    print(type(err).__name__, ":", err)

**Think 5.2** PySCF refuses: with charge 0, phenolate's atoms give 49 electrons, an odd number,
which can't all be paired (spin = 0). Here you were lucky and the mistake caused an error. For a
**dianion** like eosin Y, dropping the charge removes 2 electrons: the count stays even and
**nothing complains**. You would silently compute a different species. That is why the project
writes `charge=-2` into every xyz file and checks it against the csv. *Your answer:* …

### Exercise 5.3 — four calculations: {phenol, phenolate} × {vacuum, water}

In [ ]:
results = {}
for name, at, q in [("phenol", phenol_atoms, phenol_q), ("phenolate", phenolate_atoms, phenolate_q)]:
    for water in (False, True):
        # TODO: run single_point(...) and store (HOMO, LUMO) in results[(name, water)]
        pass

print(f"{'':10s}{'HOMO vac':>10s}{'HOMO water':>12s}{'shift':>8s}")
for name in ("phenol", "phenolate"):
    hv, hw = results[(name, False)][0], results[(name, True)][0]
    print(f"{name:10s}{hv:10.2f}{hw:12.2f}{hw-hv:8.2f}")

In [ ]:
# Check 5.3
hv_n, hw_n = results[("phenol", False)][0], results[("phenol", True)][0]
hv_a, hw_a = results[("phenolate", False)][0], results[("phenolate", True)][0]
assert abs(hw_n - hv_n) < 0.3, "the neutral molecule should barely care about the solvent"
assert hv_a > 0, "in vacuum, phenolate's HOMO should come out POSITIVE"
assert hw_a < -3, "in water it should be bound, well below zero"
print(f"OK  solvent shift: phenol {hw_n-hv_n:+.2f} eV, phenolate {hw_a-hv_a:+.2f} eV")

**Think 5.3**
- In vacuum the phenolate HOMO is **above zero**. A positive orbital energy means the electron
  would rather leave: in this calculation the anion isn't stable in vacuum. (The missing diffuse
  basis functions and B3LYP's self-interaction error both contribute, but the main message holds.)
- Water lowers the anion's HOMO by about 4.5 eV but the neutral's by less than 0.1 eV. Why does the
  solvent care so much more about the ion?
- Eosin Y²⁻ has **two** negative charges. Predict what its HOMO would look like in vacuum.
- Now explain in two sentences why the project computes every ion in ddCOSMO water.
- *Your answer:* …

---
# What you should be able to say after Lessons 3–5

1. HOMO/LUMO energies from B3LYP are good for **trends**, not absolute values. For benzene −HOMO is
   2.3 eV below the IE, while a ΔSCF energy difference reproduces it.
2. The HOMO–LUMO gap is not the optical gap. Absorption needs TDDFT, and some excitations are
   symmetry-forbidden.
3. SMILES → AddHs → ETKDG → MMFF → (symbol, xyz) → `gto.M` is the whole route from a name to a
   calculation. The project's `01_represent.py` is this plus checks and a conformer search.
4. The charge must be carried all the way to `gto.M(charge=...)`. Dropping an even number of charges
   fails **silently**.
5. Ions need a solvent model. Neutral molecules barely notice it.

---
# Solutions
*Try first! Click the cell's left bar (or the ▸ marker) to reveal a solution.*

**Solution — Exercise 3.1 — orbital energies of benzene**

In [ ]:
mol = gto.M(atom=benzene_atoms, basis="def2-svp", verbose=0)
mf = dft.RKS(mol, xc="b3lyp")
E_neutral = mf.kernel()

n_occ = mol.nelectron // 2

for k in range(n_occ - 3, n_occ + 2):
    label = "occ " if mf.mo_occ[k] > 0 else "virt"
    print(k, label, round(mf.mo_energy[k] * H2EV, 3), "eV")
HOMO = mf.mo_energy[n_occ - 1] * H2EV
LUMO = mf.mo_energy[n_occ] * H2EV

**Solution — Exercise 3.2 — ionisation energy: Koopmans vs ΔSCF**

In [ ]:
cation = gto.M(atom=benzene_atoms, basis="def2-svp", charge=1, spin=1, verbose=0)
mf_cat = dft.UKS(cation, xc="b3lyp")
E_cation = mf_cat.kernel()

IE_dSCF = (E_cation - E_neutral) * H2EV
IE_koopmans_B3LYP = -HOMO

hf = scf.RHF(mol); hf.kernel()
IE_koopmans_HF = -hf.mo_energy[n_occ - 1] * H2EV

print(f"experiment            9.24 eV")
print(f"-HOMO (B3LYP)       {IE_koopmans_B3LYP:6.2f} eV")
print(f"-HOMO (HF)          {IE_koopmans_HF:6.2f} eV")
print(f"dSCF  (B3LYP)       {IE_dSCF:6.2f} eV")

**Solution — Exercise 3.3 — look at the orbitals (optional, needs py3Dmol)**

In [ ]:
from pyscf.tools import cubegen
import py3Dmol

def show_orbital(k, iso=0.03):
    cubegen.orbital(mol, "tmp.cube", mf.mo_coeff[:, k], nx=50, ny=50, nz=50)
    cube = open("tmp.cube").read()
    v = py3Dmol.view(width=350, height=300)
    v.addModel(cube, "cube"); v.setStyle({"stick": {"radius": 0.1}})
    v.addVolumetricData(cube, "cube", {"isoval": iso, "color": "blue", "opacity": 0.8})
    v.addVolumetricData(cube, "cube", {"isoval": -iso, "color": "red", "opacity": 0.8})
    v.zoomTo(); return v

for k in (n_occ - 1, n_occ - 2, n_occ):   # HOMO, HOMO-1, LUMO
    show_orbital(k).show()

**Solution — Exercise 3.4 (bonus) — the optical gap needs TDDFT**

In [ ]:
from pyscf import tddft
td = tddft.TDDFT(mf); td.nstates = 3; td.kernel()

for i, (e, f) in enumerate(zip(td.e * H2EV, td.oscillator_strength())):
    print(f"S{i+1}: {e:.2f} eV  ({1239.84/e:.0f} nm)   oscillator strength f = {f:.3f}")
print(f"HOMO-LUMO gap: {LUMO-HOMO:.2f} eV ;  experiment: weak band 4.9 eV (254 nm), strong band ~6.9 eV")

**Solution — Exercise 4.1 — read a SMILES and check it**

In [ ]:
smi = "c1ccccc1"
m = Chem.MolFromSmiles(smi)
print("formula:", CalcMolFormula(m))
print("charge :", Chem.GetFormalCharge(m))
print("heavy atoms:", m.GetNumAtoms())
Draw.MolToImage(m, size=(250, 200))

**Solution — Exercise 4.2 — write the function smiles_to_atoms**

In [ ]:
def smiles_to_atoms(smiles, seed=42):
    """Return (atoms, charge) where atoms = [(symbol, (x, y, z)), ...] in Angstrom."""
    m = Chem.MolFromSmiles(smiles)
    q = Chem.GetFormalCharge(m)
    mh = Chem.AddHs(m)
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    AllChem.EmbedMolecule(mh, params)
    AllChem.MMFFOptimizeMolecule(mh)
    pos = mh.GetConformer().GetPositions()
    atoms = [(a.GetSymbol(), tuple(pos[i])) for i, a in enumerate(mh.GetAtoms())]
    return atoms, q

atoms, q = smiles_to_atoms("c1ccccc1")
print(len(atoms), "atoms, charge", q); print(atoms[0])

**Solution — Exercise 4.3 — hand it to PySCF**

In [ ]:
def single_point(atoms, charge=0, water=False):
    """B3LYP/def2-SVP single point. Returns (mol, mf, HOMO_eV, LUMO_eV)."""
    mol = gto.M(atom=atoms, basis="def2-svp", charge=charge, spin=0, verbose=0)
    mf = dft.RKS(mol, xc="b3lyp").density_fit()
    if water:
        mf = mf.ddCOSMO(); mf.with_solvent.eps = 78.3553
    mf.kernel()
    n_occ = mol.nelectron // 2
    return mol, mf, mf.mo_energy[n_occ - 1] * H2EV, mf.mo_energy[n_occ] * H2EV

mol_b, mf_b, h, l = single_point(atoms)
print(f"benzene (MMFF geometry): {mol_b.nelectron} electrons, {mol_b.nao} basis functions, "
      f"HOMO {h:.3f}  LUMO {l:.3f} eV")

**Solution — Exercise 4.4 — anthraquinone, and compare with the project**

In [ ]:
aq_smiles = "O=C1c2ccccc2C(=O)c2ccccc21"
aq_atoms, aq_q = smiles_to_atoms(aq_smiles)

for water in (False, True):
    m_, mf_, h_, l_ = single_point(aq_atoms, aq_q, water=water)
    print(f"{'water ' if water else 'vacuum'}  HOMO {h_:.3f}  LUMO {l_:.3f}  gap {l_-h_:.3f} eV   "
          f"({m_.nelectron} e, {m_.nao} AO)")

print("project (DFT-optimised geometry, water): HOMO -7.314  LUMO -3.041  gap 4.273 eV")

**Solution — Exercise 5.1 — build both and read the charge from the SMILES**

In [ ]:
phenol_atoms, phenol_q = smiles_to_atoms("Oc1ccccc1")
phenolate_atoms, phenolate_q = smiles_to_atoms("[O-]c1ccccc1")
print("phenol   :", len(phenol_atoms), "atoms, charge", phenol_q)
print("phenolate:", len(phenolate_atoms), "atoms, charge", phenolate_q)

**Solution — Exercise 5.2 — what happens if you forget the charge?**

In [ ]:
try:
    gto.M(atom=phenolate_atoms, basis="def2-svp", spin=0, verbose=0)
except Exception as err:
    print(type(err).__name__, ":", err)

**Solution — Exercise 5.3 — four calculations: {phenol, phenolate} × {vacuum, water}**

In [ ]:
results = {}
for name, at, q in [("phenol", phenol_atoms, phenol_q), ("phenolate", phenolate_atoms, phenolate_q)]:
    for water in (False, True):
        _, _, h_, l_ = single_point(at, q, water=water)
        results[(name, water)] = (h_, l_)

print(f"{'':10s}{'HOMO vac':>10s}{'HOMO water':>12s}{'shift':>8s}")
for name in ("phenol", "phenolate"):
    hv, hw = results[(name, False)][0], results[(name, True)][0]
    print(f"{name:10s}{hv:10.2f}{hw:12.2f}{hw-hv:8.2f}")